# **Meal Planning Apps – Customer Review Analysis**
# This notebook analyses customer reviews of meal planning apps to identify customer sentiment, common themes, strengths, and pain points.

## **EATH THIS MUCH (GOOGLE PLAY)**

### 1.1. Set Up

In [2]:
!pip install -q google-play-scraper vaderSentiment

In [3]:
from google_play_scraper import reviews, Sort
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

### 1.2. Data Collection
### 200 recent English-language Google Play reviews were collected for Eat This Much from the Australian Google Play storefront.

In [4]:
result, continuation_token = reviews(
    "com.eatthismuch",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "Eat This Much",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

df = pd.DataFrame(data)

print("Reviews collected:", len(df))
df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Eat This Much,3,"it's okay but very expensive, not worth it",2026-09-29 07:54:45,0,Google Play
1,Eat This Much,1,came back yrs later cuz I remember this app be...,2026-09-23 14:21:19,3,Google Play
2,Eat This Much,5,very good app to loss weight,2026-09-21 23:38:25,0,Google Play
3,Eat This Much,3,"Great concept. However, once you add in a coup...",2026-09-20 13:14:42,0,Google Play
4,Eat This Much,5,best meal planning app out there,2026-09-16 18:08:28,0,Google Play


### 1.3. Data Cleaning

In [5]:
df_clean = df.copy()

# Remove missing reviews
df_clean = df_clean.dropna(subset=["Review"])

# Remove empty reviews
df_clean = df_clean[
    df_clean["Review"].str.strip() != ""
]

# Remove duplicate reviews
df_clean = df_clean.drop_duplicates(
    subset=["Review"]
)

df_clean = df_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(df))
print("Reviews after cleaning:", len(df_clean))

df_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Eat This Much,3,"it's okay but very expensive, not worth it",2026-09-29 07:54:45,0,Google Play
1,Eat This Much,1,came back yrs later cuz I remember this app be...,2026-09-23 14:21:19,3,Google Play
2,Eat This Much,5,very good app to loss weight,2026-09-21 23:38:25,0,Google Play
3,Eat This Much,3,"Great concept. However, once you add in a coup...",2026-09-20 13:14:42,0,Google Play
4,Eat This Much,5,best meal planning app out there,2026-09-16 18:08:28,0,Google Play


### 1.4. Sentiment Analysis
### VADER was used to analyse the sentiment of review text. Star ratings were also incorporated to reduce obvious sentiment misclassification.

In [6]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    return analyzer.polarity_scores(text)["compound"]

def get_vader_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_clean["Sentiment Score"] = (
    df_clean["Review"].apply(get_sentiment_score)
)

df_clean["VADER Sentiment"] = (
    df_clean["Sentiment Score"].apply(get_vader_sentiment)
)

df_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,3,"it's okay but very expensive, not worth it",-0.2369,Negative
1,1,came back yrs later cuz I remember this app be...,-0.6848,Negative
2,5,very good app to loss weight,0.2247,Positive
3,3,"Great concept. However, once you add in a coup...",0.5478,Positive
4,5,best meal planning app out there,0.6369,Positive
5,5,"So easy to use, just eat the food. Amazing cus...",0.7080,Positive
6,1,I selected metric units but all recipes are st...,0.0000,Neutral
7,4,A good app but I don't use it enough to justif...,0.2382,Positive
8,1,cannot sign up with google account.,0.0000,Neutral
9,4,"I like the app, only thing id like to see is i...",0.9337,Positive


### 1.5. Hybrid Sentiment Classification
### To improve sentiment classification, star ratings were combined with VADER sentiment. Reviews with 1–2 stars were classified as Negative, 4–5 stars as Positive, while 3-star reviews retained the VADER classification.

In [7]:
def final_sentiment(row):
    if row["Rating"] <= 2:
        return "Negative"
    elif row["Rating"] >= 4:
        return "Positive"
    else:
        return row["VADER Sentiment"]

df_clean["Final Sentiment"] = df_clean.apply(
    final_sentiment,
    axis=1
)

df_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,3,"it's okay but very expensive, not worth it",Negative,Negative
1,1,came back yrs later cuz I remember this app be...,Negative,Negative
2,5,very good app to loss weight,Positive,Positive
3,3,"Great concept. However, once you add in a coup...",Positive,Positive
4,5,best meal planning app out there,Positive,Positive
5,5,"So easy to use, just eat the food. Amazing cus...",Positive,Positive
6,1,I selected metric units but all recipes are st...,Neutral,Negative
7,4,A good app but I don't use it enough to justif...,Positive,Positive
8,1,cannot sign up with google account.,Neutral,Negative
9,4,"I like the app, only thing id like to see is i...",Positive,Positive


In [8]:
df_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Positive,131
Negative,67
Neutral,2


### 1.6. Theme Analysis
### Reviews were classified into common customer themes including meal planning, recipes, nutrition, grocery shopping, personalisation, usability, pricing, and technical issues.

In [9]:
themes = {
    "Meal Planning": [
        "meal plan", "meal planning", "meal planner",
        "plan meals", "weekly plan", "plan my meals"
    ],

    "Recipes": [
        "recipe", "recipes", "recipe selection",
        "recipe options"
    ],

    "Nutrition & Goals": [
        "calorie", "calories", "macro", "macros",
        "nutrition", "protein", "weight", "diet",
        "carb", "carbs"
    ],

    "Grocery & Shopping": [
        "grocery", "groceries", "shopping list",
        "shopping", "ingredients"
    ],

    "Personalisation": [
        "customize", "customise", "custom",
        "personalized", "personalised",
        "preference", "preferences",
        "adjust", "goal", "goals"
    ],

    "Usability": [
        "easy to use", "hard to use",
        "difficult to use", "confusing",
        "interface", "navigation",
        "user friendly", "user-friendly"
    ],

    "Pricing & Subscription": [
        "price", "pricing", "expensive",
        "subscription", "premium",
        "cost", "paid", "paywall"
    ],

    "Technical Issues": [
        "bug", "bugs", "crash", "crashes",
        "error", "login", "log in",
        "sign up", "sync", "broken",
        "doesn't work", "not working"
    ]
}

In [10]:
def classify_themes(review):
    text = review.lower()
    matched_themes = []

    for theme, keywords in themes.items():
        if any(keyword in text for keyword in keywords):
            matched_themes.append(theme)

    if not matched_themes:
        matched_themes.append("Other")

    return matched_themes


df_clean["Theme"] = df_clean["Review"].apply(classify_themes)

theme_df = df_clean.explode("Theme")

theme_df["Theme"].value_counts()

,count
Theme,
Other,72
Nutrition & Goals,41
Recipes,34
Pricing & Subscription,32
Grocery & Shopping,30
Meal Planning,28
Personalisation,22
Usability,17
Technical Issues,9


### 1.7. Theme Summary

In [11]:
theme_summary = pd.crosstab(
    theme_df["Theme"],
    theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in theme_summary.columns:
        theme_summary[column] = 0

theme_summary["Total"] = (
    theme_summary["Positive"]
    + theme_summary["Negative"]
    + theme_summary["Neutral"]
)

theme_summary["Positive Rate (%)"] = (
    theme_summary["Positive"]
    / theme_summary["Total"]
    * 100
).round(1)

theme_summary["Negative Rate (%)"] = (
    theme_summary["Negative"]
    / theme_summary["Total"]
    * 100
).round(1)

theme_summary = theme_summary.sort_values(
    "Total",
    ascending=False
)

theme_summary

Final Sentiment,Negative,Neutral,Positive,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Other,24,2,46,72,63.9,33.3
Nutrition & Goals,10,0,31,41,75.6,24.4
Recipes,12,0,22,34,64.7,35.3
Pricing & Subscription,16,0,16,32,50.0,50.0
Grocery & Shopping,8,0,22,30,73.3,26.7
Meal Planning,7,0,21,28,75.0,25.0
Personalisation,7,0,15,22,68.2,31.8
Usability,2,0,15,17,88.2,11.8
Technical Issues,6,0,3,9,33.3,66.7


### 1.8 Key Findings

- 65.5% of the 200 Google Play reviews analysed were classified as positive, while 33.5% were negative.
- Usability showed the strongest positive sentiment, with 88.2% of mentions classified as positive.
- Nutrition & Goals, Meal Planning, and Grocery & Shopping also showed relatively strong positive sentiment.
- Pricing & Subscription received mixed feedback, with an equal split between positive and negative sentiment.
- Technical Issues had a high negative rate (66.7%), but the number of mentions was small (9), so this result should be interpreted cautiously.

### 1.9 Export Results

In [13]:
df_clean.to_csv(
    "eat_this_much_reviews_clean.csv",
    index=False
)

theme_summary.to_csv(
    "eat_this_much_theme_summary.csv"
)

print("Eat This Much analysis saved successfully.")

Eat This Much analysis saved successfully.


In [14]:
from google.colab import files

files.download("eat_this_much_reviews_clean.csv")
files.download("eat_this_much_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **SAMSUNG FOOD**

### 2.1 Data Collection

In [15]:
result, continuation_token = reviews(
    "com.foodient.whisk",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "Samsung Food",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

samsung_df = pd.DataFrame(data)

print("Reviews collected:", len(samsung_df))
samsung_df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Samsung Food,5,Best food app in existence,2026-10-01 12:53:37,0,Google Play
1,Samsung Food,3,Not Free. Made me do all the setup stuff only ...,2026-09-30 19:50:57,0,Google Play
2,Samsung Food,5,"great recipe organizer, planner etc. perfect a...",2026-09-30 19:04:32,0,Google Play
3,Samsung Food,5,This app is life changing for me. A place to k...,2026-09-30 10:36:16,0,Google Play
4,Samsung Food,5,best food recipe app ever!,2026-09-30 02:45:44,0,Google Play


### 2.2 Data Cleaning

In [16]:
samsung_clean = samsung_df.copy()

# Remove missing reviews
samsung_clean = samsung_clean.dropna(subset=["Review"])

# Remove empty reviews
samsung_clean = samsung_clean[
    samsung_clean["Review"].str.strip() != ""
]

# Remove duplicate reviews
samsung_clean = samsung_clean.drop_duplicates(
    subset=["Review"]
)

samsung_clean = samsung_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(samsung_df))
print("Reviews after cleaning:", len(samsung_clean))

samsung_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 199


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Samsung Food,5,Best food app in existence,2026-10-01 12:53:37,0,Google Play
1,Samsung Food,3,Not Free. Made me do all the setup stuff only ...,2026-09-30 19:50:57,0,Google Play
2,Samsung Food,5,"great recipe organizer, planner etc. perfect a...",2026-09-30 19:04:32,0,Google Play
3,Samsung Food,5,This app is life changing for me. A place to k...,2026-09-30 10:36:16,0,Google Play
4,Samsung Food,5,best food recipe app ever!,2026-09-30 02:45:44,0,Google Play


### 2.3 Sentiment Analysis

### VADER was used to analyse the sentiment of each review based on the review text.

In [17]:
samsung_clean["Sentiment Score"] = (
    samsung_clean["Review"].apply(get_sentiment_score)
)

samsung_clean["VADER Sentiment"] = (
    samsung_clean["Sentiment Score"].apply(get_vader_sentiment)
)

samsung_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,5,Best food app in existence,0.6369,Positive
1,3,Not Free. Made me do all the setup stuff only ...,0.0256,Neutral
2,5,"great recipe organizer, planner etc. perfect a...",0.8316,Positive
3,5,This app is life changing for me. A place to k...,0.8584,Positive
4,5,best food recipe app ever!,0.6696,Positive
5,5,This is a great app!! I'm able to get recipes ...,0.8715,Positive
6,5,Very Useful App 10/10. I love this app and use...,0.9623,Positive
7,5,"Me encanta, además puedo poner mis propias rec...",0.0000,Neutral
8,5,Great app!!,0.6892,Positive
9,5,I use this app almost every night,0.0000,Neutral


### 2.4 Hybrid Sentiment Classification

### Final sentiment combines star ratings with VADER sentiment. Reviews with 1–2 stars were classified as Negative, 4–5 stars as Positive, while 3-star reviews retained the VADER classification.

In [18]:
samsung_clean["Final Sentiment"] = samsung_clean.apply(
    final_sentiment,
    axis=1
)

samsung_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,5,Best food app in existence,Positive,Positive
1,3,Not Free. Made me do all the setup stuff only ...,Neutral,Neutral
2,5,"great recipe organizer, planner etc. perfect a...",Positive,Positive
3,5,This app is life changing for me. A place to k...,Positive,Positive
4,5,best food recipe app ever!,Positive,Positive
5,5,This is a great app!! I'm able to get recipes ...,Positive,Positive
6,5,Very Useful App 10/10. I love this app and use...,Positive,Positive
7,5,"Me encanta, además puedo poner mis propias rec...",Neutral,Positive
8,5,Great app!!,Positive,Positive
9,5,I use this app almost every night,Neutral,Positive


In [19]:
samsung_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Positive,129
Negative,68
Neutral,2


### 2.5 Theme Analysis

### Customer reviews were grouped into the same themes used for Eat This Much to allow consistent comparison between competitors.

In [20]:
samsung_clean["Theme"] = (
    samsung_clean["Review"].apply(classify_themes)
)

samsung_theme_df = samsung_clean.explode("Theme")

samsung_theme_df["Theme"].value_counts()

,count
Theme,
Recipes,95
Other,67
Grocery & Shopping,39
Technical Issues,21
Meal Planning,21
Usability,20
Nutrition & Goals,10
Personalisation,7
Pricing & Subscription,3


### 2.6 Theme Sentiment Summary

### Sentiment was compared across review themes to identify Samsung Food's main strengths and customer pain points.

In [21]:
samsung_theme_summary = pd.crosstab(
    samsung_theme_df["Theme"],
    samsung_theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in samsung_theme_summary.columns:
        samsung_theme_summary[column] = 0

samsung_theme_summary["Total"] = (
    samsung_theme_summary["Positive"]
    + samsung_theme_summary["Negative"]
    + samsung_theme_summary["Neutral"]
)

samsung_theme_summary["Positive Rate (%)"] = (
    samsung_theme_summary["Positive"]
    / samsung_theme_summary["Total"]
    * 100
).round(1)

samsung_theme_summary["Negative Rate (%)"] = (
    samsung_theme_summary["Negative"]
    / samsung_theme_summary["Total"]
    * 100
).round(1)

samsung_theme_summary = samsung_theme_summary.sort_values(
    "Total",
    ascending=False
)

samsung_theme_summary

Final Sentiment,Negative,Neutral,Positive,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Recipes,30,0,65,95,68.4,31.6
Other,17,2,48,67,71.6,25.4
Grocery & Shopping,13,0,26,39,66.7,33.3
Technical Issues,19,0,2,21,9.5,90.5
Meal Planning,10,0,11,21,52.4,47.6
Usability,2,0,18,20,90.0,10.0
Nutrition & Goals,2,0,8,10,80.0,20.0
Personalisation,3,0,4,7,57.1,42.9
Pricing & Subscription,2,0,1,3,33.3,66.7


### 2.7 Key Findings

- 64.8% of the 199 Google Play reviews analysed were classified as positive, while 34.2% were negative and 1.0% were neutral.
- Recipes were the most frequently discussed theme, with 95 mentions, and 68.4% of these mentions were positive.
- Usability showed the strongest positive sentiment, with 90.0% of 20 mentions classified as positive.
- Grocery & Shopping also received generally positive feedback, with 66.7% of 39 mentions classified as positive.
- Technical Issues were the clearest customer pain point, with 90.5% of 21 mentions classified as negative.
- Meal Planning received mixed feedback, with 52.4% positive and 47.6% negative sentiment.
- Pricing & Subscription had only 3 mentions, so the results for this theme should be interpreted cautiously.

### 2.8 Export Results

In [22]:
samsung_clean.to_csv(
    "samsung_food_reviews_clean.csv",
    index=False
)

samsung_theme_summary.to_csv(
    "samsung_food_theme_summary.csv"
)

print("Samsung Food analysis saved successfully.")

Samsung Food analysis saved successfully.


In [24]:
from google.colab import files

files.download("samsung_food_reviews_clean.csv")
files.download("samsung_food_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **EMEALS**

### 3.1 Data Collection

In [25]:
result, continuation_token = reviews(
    "com.emeals",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "eMeals",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

emeals_df = pd.DataFrame(data)

print("Reviews collected:", len(emeals_df))
emeals_df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,eMeals,2,Freezes Constantly! The convenience of the mea...,2026-09-30 02:08:49,0,Google Play
1,eMeals,1,I can't figure out how to make a new meal plan...,2026-09-27 19:48:10,0,Google Play
2,eMeals,2,"great concept, but the app is very glitchy, me...",2026-09-27 13:08:19,0,Google Play
3,eMeals,1,"I did not order, I don't have an account and d...",2026-09-24 12:44:41,0,Google Play
4,eMeals,1,THE APP IS SO SLOW. I JUST SPENT 20 MINUTES JU...,2026-09-23 15:37:17,0,Google Play


### 3.2 Data Cleaning

In [26]:
emeals_clean = emeals_df.copy()

emeals_clean = emeals_clean.dropna(subset=["Review"])

emeals_clean = emeals_clean[
    emeals_clean["Review"].str.strip() != ""
]

emeals_clean = emeals_clean.drop_duplicates(
    subset=["Review"]
)

emeals_clean = emeals_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(emeals_df))
print("Reviews after cleaning:", len(emeals_clean))

emeals_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,eMeals,2,Freezes Constantly! The convenience of the mea...,2026-09-30 02:08:49,0,Google Play
1,eMeals,1,I can't figure out how to make a new meal plan...,2026-09-27 19:48:10,0,Google Play
2,eMeals,2,"great concept, but the app is very glitchy, me...",2026-09-27 13:08:19,0,Google Play
3,eMeals,1,"I did not order, I don't have an account and d...",2026-09-24 12:44:41,0,Google Play
4,eMeals,1,THE APP IS SO SLOW. I JUST SPENT 20 MINUTES JU...,2026-09-23 15:37:17,0,Google Play


### 3.3 Sentiment Analysis

### VADER was used to analyse the sentiment of each review based on the review text.

In [27]:
emeals_clean["Sentiment Score"] = (
    emeals_clean["Review"].apply(get_sentiment_score)
)

emeals_clean["VADER Sentiment"] = (
    emeals_clean["Sentiment Score"].apply(get_vader_sentiment)
)

emeals_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,2,Freezes Constantly! The convenience of the mea...,0.3056,Positive
1,1,I can't figure out how to make a new meal plan...,0.0772,Positive
2,2,"great concept, but the app is very glitchy, me...",0.3716,Positive
3,1,"I did not order, I don't have an account and d...",-0.1280,Negative
4,1,THE APP IS SO SLOW. I JUST SPENT 20 MINUTES JU...,-0.2960,Negative
5,1,Rotten cancelation policies. Did not help me f...,-0.6083,Negative
6,4,So slow and glitchy! Recipes are generally goo...,-0.5848,Negative
7,1,"This app, while useful and helpful, does not s...",0.5975,Positive
8,2,Very convenient and saves time for the most pa...,0.1027,Positive
9,1,Recipes are only good for people that think Mc...,0.7003,Positive


### 3.4 Hybrid Sentiment Classification

In [28]:
emeals_clean["Final Sentiment"] = emeals_clean.apply(
    final_sentiment,
    axis=1
)

emeals_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,2,Freezes Constantly! The convenience of the mea...,Positive,Negative
1,1,I can't figure out how to make a new meal plan...,Positive,Negative
2,2,"great concept, but the app is very glitchy, me...",Positive,Negative
3,1,"I did not order, I don't have an account and d...",Negative,Negative
4,1,THE APP IS SO SLOW. I JUST SPENT 20 MINUTES JU...,Negative,Negative
5,1,Rotten cancelation policies. Did not help me f...,Negative,Negative
6,4,So slow and glitchy! Recipes are generally goo...,Negative,Positive
7,1,"This app, while useful and helpful, does not s...",Positive,Negative
8,2,Very convenient and saves time for the most pa...,Positive,Negative
9,1,Recipes are only good for people that think Mc...,Positive,Negative


In [29]:
emeals_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Negative,124
Positive,70
Neutral,6


### 3.5 Theme Analysis

In [30]:
emeals_clean["Theme"] = (
    emeals_clean["Review"].apply(classify_themes)
)

emeals_theme_df = emeals_clean.explode("Theme")

emeals_theme_df["Theme"].value_counts()

,count
Theme,
Other,78
Recipes,57
Grocery & Shopping,43
Technical Issues,35
Pricing & Subscription,25
Meal Planning,18
Personalisation,6
Usability,6
Nutrition & Goals,1


### 3.6 Theme Sentiment Summary

In [31]:
emeals_theme_summary = pd.crosstab(
    emeals_theme_df["Theme"],
    emeals_theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in emeals_theme_summary.columns:
        emeals_theme_summary[column] = 0

emeals_theme_summary["Total"] = (
    emeals_theme_summary["Positive"]
    + emeals_theme_summary["Negative"]
    + emeals_theme_summary["Neutral"]
)

emeals_theme_summary["Positive Rate (%)"] = (
    emeals_theme_summary["Positive"]
    / emeals_theme_summary["Total"]
    * 100
).round(1)

emeals_theme_summary["Negative Rate (%)"] = (
    emeals_theme_summary["Negative"]
    / emeals_theme_summary["Total"]
    * 100
).round(1)

emeals_theme_summary = emeals_theme_summary.sort_values(
    "Total",
    ascending=False
)

emeals_theme_summary

Final Sentiment,Negative,Neutral,Positive,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Other,53,4,21,78,26.9,67.9
Recipes,29,1,27,57,47.4,50.9
Grocery & Shopping,20,0,23,43,53.5,46.5
Technical Issues,28,1,6,35,17.1,80.0
Pricing & Subscription,21,0,4,25,16.0,84.0
Meal Planning,8,0,10,18,55.6,44.4
Personalisation,1,0,5,6,83.3,16.7
Usability,2,0,4,6,66.7,33.3
Nutrition & Goals,0,0,1,1,100.0,0.0


### 3.7 Key Findings

- 35.0% of the 200 Google Play reviews analysed were classified as positive, while 62.0% were negative and 3.0% were neutral.
- Recipes were the most frequently discussed specific theme, with 57 mentions, but sentiment was mixed: 47.4% positive and 50.9% negative.
- Grocery & Shopping received relatively balanced feedback, with 53.5% positive and 46.5% negative sentiment across 43 mentions.
- Technical Issues were a major customer pain point, with 80.0% of 35 mentions classified as negative.
- Pricing & Subscription showed the strongest negative sentiment among substantial themes, with 84.0% of 25 mentions classified as negative.
- Meal Planning received mixed but slightly more positive feedback, with 55.6% positive and 44.4% negative sentiment.
- Personalisation, Usability, and Nutrition & Goals had relatively few mentions, so these results should be interpreted cautiously.

 ### 3.8 Export Results

In [32]:
emeals_clean.to_csv(
    "emeals_reviews_clean.csv",
    index=False
)

emeals_theme_summary.to_csv(
    "emeals_theme_summary.csv"
)

print("eMeals analysis saved successfully.")

eMeals analysis saved successfully.


In [33]:
from google.colab import files

files.download("emeals_reviews_clean.csv")
files.download("emeals_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **PLAN TO EAT**

### 4.1 Data Collection

In [34]:
result, continuation_token = reviews(
    "com.plantoeat.mobile",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "Plan to Eat",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

plantoeat_df = pd.DataFrame(data)

print("Reviews collected:", len(plantoeat_df))
plantoeat_df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Plan to Eat,1,"As soon as I started it, it said I was startin...",2026-09-22 19:32:28,0,Google Play
1,Plan to Eat,5,So helpful for a busy family. Two years later ...,2026-09-22 13:03:39,0,Google Play
2,Plan to Eat,4,"Love this app honestly, I use it for all my re...",2026-09-17 15:29:39,0,Google Play
3,Plan to Eat,1,was great until I was locked out and forced to...,2026-09-01 00:18:36,2,Google Play
4,Plan to Eat,1,Gives too many receipes that my family won't eat.,2026-08-31 16:57:31,0,Google Play


### 4.2 Data Cleaning

In [35]:
plantoeat_clean = plantoeat_df.copy()

plantoeat_clean = plantoeat_clean.dropna(subset=["Review"])

plantoeat_clean = plantoeat_clean[
    plantoeat_clean["Review"].str.strip() != ""
]

plantoeat_clean = plantoeat_clean.drop_duplicates(
    subset=["Review"]
)

plantoeat_clean = plantoeat_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(plantoeat_df))
print("Reviews after cleaning:", len(plantoeat_clean))

plantoeat_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Plan to Eat,1,"As soon as I started it, it said I was startin...",2026-09-22 19:32:28,0,Google Play
1,Plan to Eat,5,So helpful for a busy family. Two years later ...,2026-09-22 13:03:39,0,Google Play
2,Plan to Eat,4,"Love this app honestly, I use it for all my re...",2026-09-17 15:29:39,0,Google Play
3,Plan to Eat,1,was great until I was locked out and forced to...,2026-09-01 00:18:36,2,Google Play
4,Plan to Eat,1,Gives too many receipes that my family won't eat.,2026-08-31 16:57:31,0,Google Play


### 4.3 Sentiment Analysis

In [36]:
plantoeat_clean["Sentiment Score"] = (
    plantoeat_clean["Review"].apply(get_sentiment_score)
)

plantoeat_clean["VADER Sentiment"] = (
    plantoeat_clean["Sentiment Score"].apply(get_vader_sentiment)
)

plantoeat_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,1,"As soon as I started it, it said I was startin...",0.7650,Positive
1,5,So helpful for a busy family. Two years later ...,0.8218,Positive
2,4,"Love this app honestly, I use it for all my re...",0.8020,Positive
3,1,was great until I was locked out and forced to...,-0.3806,Negative
4,1,Gives too many receipes that my family won't eat.,0.0000,Neutral
5,1,meal planner apps should not be subscription b...,0.0000,Neutral
6,5,it would be really cool if there was a widget ...,0.3804,Positive
7,5,This is the one. I spent years-YEARS!-trying o...,0.9620,Positive
8,5,This has been the best app ever! I have been u...,0.8988,Positive
9,5,Great app for saving recipes and being able to...,0.9317,Positive


### 4.4 Hybrid Sentiment Classification

In [37]:
plantoeat_clean["Final Sentiment"] = plantoeat_clean.apply(
    final_sentiment,
    axis=1
)

plantoeat_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,1,"As soon as I started it, it said I was startin...",Positive,Negative
1,5,So helpful for a busy family. Two years later ...,Positive,Positive
2,4,"Love this app honestly, I use it for all my re...",Positive,Positive
3,1,was great until I was locked out and forced to...,Negative,Negative
4,1,Gives too many receipes that my family won't eat.,Neutral,Negative
5,1,meal planner apps should not be subscription b...,Neutral,Negative
6,5,it would be really cool if there was a widget ...,Positive,Positive
7,5,This is the one. I spent years-YEARS!-trying o...,Positive,Positive
8,5,This has been the best app ever! I have been u...,Positive,Positive
9,5,Great app for saving recipes and being able to...,Positive,Positive


In [38]:
plantoeat_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Positive,188
Negative,12


### 4.5 Theme Analysis

In [39]:
plantoeat_clean["Theme"] = (
    plantoeat_clean["Review"].apply(classify_themes)
)

plantoeat_theme_df = plantoeat_clean.explode("Theme")

plantoeat_theme_df["Theme"].value_counts()

,count
Theme,
Recipes,120
Grocery & Shopping,79
Meal Planning,49
Other,37
Pricing & Subscription,27
Usability,24
Personalisation,15
Nutrition & Goals,7
Technical Issues,6


### 4.6 Theme Sentiment Summary

In [40]:
plantoeat_theme_summary = pd.crosstab(
    plantoeat_theme_df["Theme"],
    plantoeat_theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in plantoeat_theme_summary.columns:
        plantoeat_theme_summary[column] = 0

plantoeat_theme_summary["Total"] = (
    plantoeat_theme_summary["Positive"]
    + plantoeat_theme_summary["Negative"]
    + plantoeat_theme_summary["Neutral"]
)

plantoeat_theme_summary["Positive Rate (%)"] = (
    plantoeat_theme_summary["Positive"]
    / plantoeat_theme_summary["Total"]
    * 100
).round(1)

plantoeat_theme_summary["Negative Rate (%)"] = (
    plantoeat_theme_summary["Negative"]
    / plantoeat_theme_summary["Total"]
    * 100
).round(1)

plantoeat_theme_summary = plantoeat_theme_summary.sort_values(
    "Total",
    ascending=False
)

plantoeat_theme_summary

Final Sentiment,Negative,Positive,Neutral,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Recipes,2,118,0,120,98.3,1.7
Grocery & Shopping,0,79,0,79,100.0,0.0
Meal Planning,1,48,0,49,98.0,2.0
Other,2,35,0,37,94.6,5.4
Pricing & Subscription,8,19,0,27,70.4,29.6
Usability,0,24,0,24,100.0,0.0
Personalisation,0,15,0,15,100.0,0.0
Nutrition & Goals,0,7,0,7,100.0,0.0
Technical Issues,2,4,0,6,66.7,33.3


### 4.7 Key Findings

- 94.0% of the 200 Google Play reviews analysed were classified as positive, while 6.0% were negative.
- Recipes were the most frequently discussed theme, with 120 mentions, and 98.3% of these mentions were positive.
- Grocery & Shopping received entirely positive sentiment across 79 mentions, making it one of the strongest aspects of the app in this sample.
- Meal Planning also received highly positive feedback, with 98.0% of 49 mentions classified as positive.
- Usability and Personalisation received 100% positive sentiment, although they had smaller sample sizes of 24 and 15 mentions respectively.
- Pricing & Subscription showed more mixed feedback, with 29.6% of 27 mentions classified as negative.
- Technical Issues had the highest negative rate at 33.3%, but this theme appeared in only 6 mentions and should therefore be interpreted cautiously.

### 4.8 Export Results

In [41]:
plantoeat_clean.to_csv(
    "plan_to_eat_reviews_clean.csv",
    index=False
)

plantoeat_theme_summary.to_csv(
    "plan_to_eat_theme_summary.csv"
)

print("Plan to Eat analysis saved successfully.")

Plan to Eat analysis saved successfully.


In [42]:
from google.colab import files

files.download("plan_to_eat_reviews_clean.csv")
files.download("plan_to_eat_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **PAPRIKA**

### 5.1 Data Collection

In [43]:
result, continuation_token = reviews(
    "com.hindsightlabs.paprika.android.v3",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "Paprika",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

paprika_df = pd.DataFrame(data)

print("Reviews collected:", len(paprika_df))
paprika_df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Paprika,5,Love this app. Makes saving recipes easy,2026-10-01 17:15:09,0,Google Play
1,Paprika,5,This app has become indispensable in my home. ...,2026-10-01 00:33:47,0,Google Play
2,Paprika,5,best recipe app I have ever had. I have tried ...,2026-09-28 15:12:40,0,Google Play
3,Paprika,5,love how it keeps all your recipes simply in o...,2026-09-28 04:30:04,0,Google Play
4,Paprika,4,This is a great app. Has all the features I ne...,2026-09-28 00:13:17,0,Google Play


### 5.2 Data Cleaning

In [44]:
paprika_clean = paprika_df.copy()

paprika_clean = paprika_clean.dropna(subset=["Review"])

paprika_clean = paprika_clean[
    paprika_clean["Review"].str.strip() != ""
]

paprika_clean = paprika_clean.drop_duplicates(
    subset=["Review"]
)

paprika_clean = paprika_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(paprika_df))
print("Reviews after cleaning:", len(paprika_clean))

paprika_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Paprika,5,Love this app. Makes saving recipes easy,2026-10-01 17:15:09,0,Google Play
1,Paprika,5,This app has become indispensable in my home. ...,2026-10-01 00:33:47,0,Google Play
2,Paprika,5,best recipe app I have ever had. I have tried ...,2026-09-28 15:12:40,0,Google Play
3,Paprika,5,love how it keeps all your recipes simply in o...,2026-09-28 04:30:04,0,Google Play
4,Paprika,4,This is a great app. Has all the features I ne...,2026-09-28 00:13:17,0,Google Play


### 5.3 Sentiment Analysis

In [45]:
paprika_clean["Sentiment Score"] = (
    paprika_clean["Review"].apply(get_sentiment_score)
)

paprika_clean["VADER Sentiment"] = (
    paprika_clean["Sentiment Score"].apply(get_vader_sentiment)
)

paprika_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,5,Love this app. Makes saving recipes easy,0.7964,Positive
1,5,This app has become indispensable in my home. ...,-0.4374,Negative
2,5,best recipe app I have ever had. I have tried ...,0.9200,Positive
3,5,love how it keeps all your recipes simply in o...,0.7964,Positive
4,4,This is a great app. Has all the features I ne...,0.8401,Positive
5,5,I paid for the premium version of this app and...,0.3581,Positive
6,5,LOVE this app. Had it for years and it's the o...,0.7371,Positive
7,5,best recipe app out there. completely worth th...,0.7501,Positive
8,5,"The perfect app. Does exactly what it says, no...",0.8219,Positive
9,5,Excellent app!!,0.6467,Positive


### 5.4 Hybrid Sentiment Classification

In [46]:
paprika_clean["Final Sentiment"] = paprika_clean.apply(
    final_sentiment,
    axis=1
)

paprika_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,5,Love this app. Makes saving recipes easy,Positive,Positive
1,5,This app has become indispensable in my home. ...,Negative,Positive
2,5,best recipe app I have ever had. I have tried ...,Positive,Positive
3,5,love how it keeps all your recipes simply in o...,Positive,Positive
4,4,This is a great app. Has all the features I ne...,Positive,Positive
5,5,I paid for the premium version of this app and...,Positive,Positive
6,5,LOVE this app. Had it for years and it's the o...,Positive,Positive
7,5,best recipe app out there. completely worth th...,Positive,Positive
8,5,"The perfect app. Does exactly what it says, no...",Positive,Positive
9,5,Excellent app!!,Positive,Positive


In [47]:
paprika_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Positive,197
Negative,3


### 5.5 Theme Analysis

In [48]:
paprika_clean["Theme"] = (
    paprika_clean["Review"].apply(classify_themes)
)

paprika_theme_df = paprika_clean.explode("Theme")

paprika_theme_df["Theme"].value_counts()

,count
Theme,
Recipes,128
Other,49
Grocery & Shopping,44
Pricing & Subscription,31
Usability,24
Meal Planning,18
Technical Issues,9
Nutrition & Goals,4
Personalisation,2


### 5.6 Theme Sentiment Summary

In [49]:
paprika_theme_summary = pd.crosstab(
    paprika_theme_df["Theme"],
    paprika_theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in paprika_theme_summary.columns:
        paprika_theme_summary[column] = 0

paprika_theme_summary["Total"] = (
    paprika_theme_summary["Positive"]
    + paprika_theme_summary["Negative"]
    + paprika_theme_summary["Neutral"]
)

paprika_theme_summary["Positive Rate (%)"] = (
    paprika_theme_summary["Positive"]
    / paprika_theme_summary["Total"]
    * 100
).round(1)

paprika_theme_summary["Negative Rate (%)"] = (
    paprika_theme_summary["Negative"]
    / paprika_theme_summary["Total"]
    * 100
).round(1)

paprika_theme_summary = paprika_theme_summary.sort_values(
    "Total",
    ascending=False
)

paprika_theme_summary

Final Sentiment,Negative,Positive,Neutral,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Recipes,1,127,0,128,99.2,0.8
Other,1,48,0,49,98.0,2.0
Grocery & Shopping,0,44,0,44,100.0,0.0
Pricing & Subscription,1,30,0,31,96.8,3.2
Usability,0,24,0,24,100.0,0.0
Meal Planning,0,18,0,18,100.0,0.0
Technical Issues,0,9,0,9,100.0,0.0
Nutrition & Goals,0,4,0,4,100.0,0.0
Personalisation,0,2,0,2,100.0,0.0


### 5.7 Key Findings

- 98.5% of the 200 Google Play reviews analysed were classified as positive, while only 1.5% were negative.
- Recipes were the most frequently discussed theme, with 128 mentions, and 99.2% of these mentions were positive.
- Grocery & Shopping received entirely positive sentiment across 44 mentions.
- Pricing & Subscription was also viewed positively, with 96.8% of 31 mentions classified as positive.
- Usability and Meal Planning received 100% positive sentiment across 24 and 18 mentions respectively.
- Technical Issues had 9 mentions, all classified as positive; however, the review text should be examined before interpreting these mentions as evidence of technical performance.
- Nutrition & Goals and Personalisation also showed positive sentiment, but their small number of mentions means these results should be interpreted cautiously.

### 5.8 Export Results

In [50]:
paprika_clean.to_csv(
    "paprika_reviews_clean.csv",
    index=False
)

paprika_theme_summary.to_csv(
    "paprika_theme_summary.csv"
)

print("Paprika analysis saved successfully.")

Paprika analysis saved successfully.


In [51]:
from google.colab import files

files.download("paprika_reviews_clean.csv")
files.download("paprika_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **MEALIME**

### 6.1 Data Collection

In [52]:
result, continuation_token = reviews(
    "com.mealime",
    lang="en",
    country="au",
    sort=Sort.NEWEST,
    count=200
)

data = []

for review in result:
    data.append({
        "Competitor": "Mealime",
        "Rating": review["score"],
        "Review": review["content"],
        "Date": review["at"],
        "Thumbs Up": review["thumbsUpCount"],
        "Source": "Google Play"
    })

mealime_df = pd.DataFrame(data)

print("Reviews collected:", len(mealime_df))
mealime_df.head()

Reviews collected: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Mealime,5,I love being inspired by all the yummy recipes...,2026-09-30 18:01:39,0,Google Play
1,Mealime,5,I've loved this app. are you going to add Walm...,2026-09-28 23:19:09,0,Google Play
2,Mealime,1,my heart is broken. your meals hub bs inside o...,2026-09-28 20:09:35,1,Google Play
3,Mealime,2,Shame it's stopping 😕 I've been using this app...,2026-09-28 06:53:55,1,Google Play
4,Mealime,1,Please update your allergy list to have specif...,2026-09-27 21:19:31,0,Google Play


### 6.2 Data Cleaning

In [53]:
mealime_clean = mealime_df.copy()

mealime_clean = mealime_clean.dropna(subset=["Review"])

mealime_clean = mealime_clean[
    mealime_clean["Review"].str.strip() != ""
]

mealime_clean = mealime_clean.drop_duplicates(
    subset=["Review"]
)

mealime_clean = mealime_clean.reset_index(drop=True)

print("Reviews before cleaning:", len(mealime_df))
print("Reviews after cleaning:", len(mealime_clean))

mealime_clean.head()

Reviews before cleaning: 200
Reviews after cleaning: 200


,Competitor,Rating,Review,Date,Thumbs Up,Source
0,Mealime,5,I love being inspired by all the yummy recipes...,2026-09-30 18:01:39,0,Google Play
1,Mealime,5,I've loved this app. are you going to add Walm...,2026-09-28 23:19:09,0,Google Play
2,Mealime,1,my heart is broken. your meals hub bs inside o...,2026-09-28 20:09:35,1,Google Play
3,Mealime,2,Shame it's stopping 😕 I've been using this app...,2026-09-28 06:53:55,1,Google Play
4,Mealime,1,Please update your allergy list to have specif...,2026-09-27 21:19:31,0,Google Play


### 6.3 Sentiment Analysis

In [54]:
mealime_clean["Sentiment Score"] = (
    mealime_clean["Review"].apply(get_sentiment_score)
)

mealime_clean["VADER Sentiment"] = (
    mealime_clean["Sentiment Score"].apply(get_vader_sentiment)
)

mealime_clean[
    ["Rating", "Review", "Sentiment Score", "VADER Sentiment"]
].head(10)

,Rating,Review,Sentiment Score,VADER Sentiment
0,5,I love being inspired by all the yummy recipes...,0.8957,Positive
1,5,I've loved this app. are you going to add Walm...,0.5994,Positive
2,1,my heart is broken. your meals hub bs inside o...,-0.7003,Negative
3,2,Shame it's stopping 😕 I've been using this app...,-0.8755,Negative
4,1,Please update your allergy list to have specif...,0.0258,Neutral
5,5,"Real shame app is shutting down, fantastic for...",0.7096,Positive
6,5,nice,0.4215,Positive
7,4,genuinely I can not express how this makes my ...,0.0460,Neutral
8,1,Why would you shut down? Stupidest decision. I...,-0.8903,Negative
9,4,Mealime has been wonderful for the five years ...,0.9199,Positive


### 6.4 Hybrid Sentiment Classification

In [55]:
mealime_clean["Final Sentiment"] = mealime_clean.apply(
    final_sentiment,
    axis=1
)

mealime_clean[
    ["Rating", "Review", "VADER Sentiment", "Final Sentiment"]
].head(10)

,Rating,Review,VADER Sentiment,Final Sentiment
0,5,I love being inspired by all the yummy recipes...,Positive,Positive
1,5,I've loved this app. are you going to add Walm...,Positive,Positive
2,1,my heart is broken. your meals hub bs inside o...,Negative,Negative
3,2,Shame it's stopping 😕 I've been using this app...,Negative,Negative
4,1,Please update your allergy list to have specif...,Neutral,Negative
5,5,"Real shame app is shutting down, fantastic for...",Positive,Positive
6,5,nice,Positive,Positive
7,4,genuinely I can not express how this makes my ...,Neutral,Positive
8,1,Why would you shut down? Stupidest decision. I...,Negative,Negative
9,4,Mealime has been wonderful for the five years ...,Positive,Positive


In [56]:
mealime_clean["Final Sentiment"].value_counts()

,count
Final Sentiment,
Positive,143
Negative,55
Neutral,2


### 6.5 Theme Analysis

In [57]:
mealime_clean["Theme"] = (
    mealime_clean["Review"].apply(classify_themes)
)

mealime_theme_df = mealime_clean.explode("Theme")

mealime_theme_df["Theme"].value_counts()

,count
Theme,
Grocery & Shopping,70
Recipes,68
Other,64
Meal Planning,33
Usability,20
Technical Issues,18
Nutrition & Goals,16
Pricing & Subscription,15
Personalisation,10


### 6.6 Theme Sentiment Summary

In [58]:
mealime_theme_summary = pd.crosstab(
    mealime_theme_df["Theme"],
    mealime_theme_df["Final Sentiment"]
)

for column in ["Positive", "Negative", "Neutral"]:
    if column not in mealime_theme_summary.columns:
        mealime_theme_summary[column] = 0

mealime_theme_summary["Total"] = (
    mealime_theme_summary["Positive"]
    + mealime_theme_summary["Negative"]
    + mealime_theme_summary["Neutral"]
)

mealime_theme_summary["Positive Rate (%)"] = (
    mealime_theme_summary["Positive"]
    / mealime_theme_summary["Total"]
    * 100
).round(1)

mealime_theme_summary["Negative Rate (%)"] = (
    mealime_theme_summary["Negative"]
    / mealime_theme_summary["Total"]
    * 100
).round(1)

mealime_theme_summary = mealime_theme_summary.sort_values(
    "Total",
    ascending=False
)

mealime_theme_summary

Final Sentiment,Negative,Neutral,Positive,Total,Positive Rate (%),Negative Rate (%)
Theme,,,,,,
Grocery & Shopping,9,1,60,70,85.7,12.9
Recipes,15,0,53,68,77.9,22.1
Other,20,1,43,64,67.2,31.2
Meal Planning,8,0,25,33,75.8,24.2
Usability,3,0,17,20,85.0,15.0
Technical Issues,11,1,6,18,33.3,61.1
Nutrition & Goals,4,0,12,16,75.0,25.0
Pricing & Subscription,2,0,13,15,86.7,13.3
Personalisation,4,0,6,10,60.0,40.0


### 6.7 Key Findings

- 71.5% of the 200 Google Play reviews analysed were classified as positive, while 27.5% were negative and 1.0% were neutral.
- Grocery & Shopping was the most frequently discussed theme, with 70 mentions, and 85.7% of these mentions were positive.
- Recipes were also frequently discussed, with 68 mentions and 77.9% positive sentiment.
- Meal Planning received generally positive feedback, with 75.8% of 33 mentions classified as positive.
- Usability showed strong positive sentiment, with 85.0% of 20 mentions classified as positive.
- Technical Issues were the clearest customer pain point, with 61.1% of 18 mentions classified as negative.
- Pricing & Subscription received largely positive sentiment, with 86.7% of 15 mentions classified as positive.
- These findings should be interpreted cautiously because recent reviews may have been influenced by Mealime's announced service closure.

### 6.8 Export Results

In [59]:
mealime_clean.to_csv(
    "mealime_reviews_clean.csv",
    index=False
)

mealime_theme_summary.to_csv(
    "mealime_theme_summary.csv"
)

print("Mealime analysis saved successfully.")

Mealime analysis saved successfully.


In [60]:
from google.colab import files

files.download("mealime_reviews_clean.csv")
files.download("mealime_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Cross-Competitor Summary**

### 7.1 Overall Sentiment Comparison

In [61]:
competitor_datasets = {
    "Eat This Much": df_clean,
    "Samsung Food": samsung_clean,
    "eMeals": emeals_clean,
    "Plan to Eat": plantoeat_clean,
    "Paprika": paprika_clean,
    "Mealime": mealime_clean
}

summary_data = []

for competitor, dataset in competitor_datasets.items():

    total = len(dataset)

    sentiment_counts = (
        dataset["Final Sentiment"]
        .value_counts()
    )

    positive = sentiment_counts.get("Positive", 0)
    negative = sentiment_counts.get("Negative", 0)
    neutral = sentiment_counts.get("Neutral", 0)

    summary_data.append({
        "Competitor": competitor,
        "Reviews Analysed": total,
        "Positive Reviews": positive,
        "Negative Reviews": negative,
        "Neutral Reviews": neutral,
        "Positive %": round(positive / total * 100, 1),
        "Negative %": round(negative / total * 100, 1),
        "Neutral %": round(neutral / total * 100, 1)
    })

competitor_summary = pd.DataFrame(summary_data)

competitor_summary

,Competitor,Reviews Analysed,Positive Reviews,Negative Reviews,Neutral Reviews,Positive %,Negative %,Neutral %
0,Eat This Much,200,131,67,2,65.5,33.5,1.0
1,Samsung Food,199,129,68,2,64.8,34.2,1.0
2,eMeals,200,70,124,6,35.0,62.0,3.0
3,Plan to Eat,200,188,12,0,94.0,6.0,0.0
4,Paprika,200,197,3,0,98.5,1.5,0.0
5,Mealime,200,143,55,2,71.5,27.5,1.0


### 7.2 Theme Comparison Dataset
Theme-level sentiment results were combined into a single dataset to support cross-competitor comparison and dashboard development.

In [62]:
theme_summaries = {
    "Eat This Much": theme_summary,
    "Samsung Food": samsung_theme_summary,
    "eMeals": emeals_theme_summary,
    "Plan to Eat": plantoeat_theme_summary,
    "Paprika": paprika_theme_summary,
    "Mealime": mealime_theme_summary
}

combined_theme_data = []

for competitor, summary in theme_summaries.items():

    temp = summary.reset_index().copy()
    temp["Competitor"] = competitor

    combined_theme_data.append(temp)

competitor_theme_summary = pd.concat(
    combined_theme_data,
    ignore_index=True
)

competitor_theme_summary = competitor_theme_summary[
    [
        "Competitor",
        "Theme",
        "Positive",
        "Negative",
        "Neutral",
        "Total",
        "Positive Rate (%)",
        "Negative Rate (%)"
    ]
]

competitor_theme_summary.head(20)

Final Sentiment,Competitor,Theme,Positive,Negative,Neutral,Total,Positive Rate (%),Negative Rate (%)
0,Eat This Much,Other,46,24,2,72,63.9,33.3
1,Eat This Much,Nutrition & Goals,31,10,0,41,75.6,24.4
2,Eat This Much,Recipes,22,12,0,34,64.7,35.3
3,Eat This Much,Pricing & Subscription,16,16,0,32,50.0,50.0
4,Eat This Much,Grocery & Shopping,22,8,0,30,73.3,26.7
5,Eat This Much,Meal Planning,21,7,0,28,75.0,25.0
6,Eat This Much,Personalisation,15,7,0,22,68.2,31.8
7,Eat This Much,Usability,15,2,0,17,88.2,11.8
8,Eat This Much,Technical Issues,3,6,0,9,33.3,66.7
9,Samsung Food,Recipes,65,30,0,95,68.4,31.6


### 7.3 Export Combined Results

The overall competitor sentiment summary and theme-level comparison dataset were exported for use in Excel and Power BI.

In [63]:
competitor_summary.to_csv(
    "competitor_review_summary.csv",
    index=False
)

competitor_theme_summary.to_csv(
    "competitor_theme_summary.csv",
    index=False
)

print("Cross-competitor datasets saved successfully.")

Cross-competitor datasets saved successfully.


### 7.4 Download Combined Results

In [64]:
from google.colab import files

files.download("competitor_review_summary.csv")
files.download("competitor_theme_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>